# 第 7 节：时序差分（TD）学习

---

## 📍 本节位置

```
MC (06) → **TD Learning (07)** → SARSA/Q-Learning (08) → ...
              ↑
          你在这里
```

TD 学习是 RL 中**最核心**的思想之一。它结合了 MC（从经验学习）和 DP（bootstrapping）的优点。

---

## 🎯 学习目标

1. 理解 TD(0) 的更新规则及其推导
2. 掌握 bootstrapping 的含义
3. 对比 MC 和 TD 的 bias-variance trade-off
4. 理解 n-step TD 和 TD(λ) 的直觉
5. 在随机游走环境中验证 TD 的优势


## 1. TD(0)：一步时序差分

### 核心思想

MC 更新（等 episode 结束）：
$$V(S_t) \leftarrow V(S_t) + \alpha [G_t - V(S_t)]$$

TD(0) 更新（不等待，立即 bootstrap）：
$$V(S_t) \leftarrow V(S_t) + \alpha [R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$

### 关键术语

- **TD Target**：$R_{t+1} + \gamma V(S_{t+1})$ — 用当前 V 估计的期望回报
- **TD Error**：$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ — 实际与预期的差距
- **Bootstrapping**：用 $V(S_{t+1})$（一个估计值）来更新 $V(S_t)$（另一个估计值）

### 与 MC 的关系

把 TD target 展开：

$$
\begin{aligned}
R_{t+1} + \gamma V(S_{t+1}) &= R_{t+1} + \gamma(R_{t+2} + \gamma V(S_{t+2})) \\
&= R_{t+1} + \gamma R_{t+2} + \gamma^2 V(S_{t+2}) \\
&= \cdots \\
&= R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{T-t-1} R_T
\end{aligned}
$$

当 episode 终止时 $V(S_T)=0$，TD target 完全展开就是 **MC return** $G_t$！

**所以 TD 可以看作"提前截断的 MC"**——用当前的 V 估计替代未知的未来。


## 2. MC vs TD：Bias-Variance Trade-off

### 直观理解

| 方法 | Target | Bias | Variance | 更新时机 |
|------|--------|------|----------|----------|
| **MC** | $G_t$（实际回报） | 0 | 大 | Episode 结束后 |
| **TD(0)** | $R_{t+1}+\gamma V(S_{t+1})$ | 有（来自 V 的误差） | 小 | 每步 |

### 为什么 TD 方差通常更小？

MC 的 $G_t$ 累积了所有后续步骤的随机性。其完整方差包括各步奖励的方差以及它们之间的协方差：

$$\text{Var}[G_t] = \sum_{k=0}^{\infty} \gamma^{2k} \text{Var}[R_{t+k+1}] + 2\sum_{i<j} \gamma^{i+j} \text{Cov}[R_{t+i+1}, R_{t+j+1}]$$

而 TD(0) 只涉及一步转移的随机性：

$$\text{Var}[R_{t+1} + \gamma V(S_{t+1})] = \text{Var}[R_{t+1}] + \gamma^2 \text{Var}[V(S_{t+1})] + 2\gamma \text{Cov}[R_{t+1}, V(S_{t+1})]$$

**关键理解**: 虽然 $V(s)$ 对给定状态 $s$ 是确定性函数，但在随机转移或随机策略下，$S_{t+1}$ 本身是随机变量，因此 $V(S_{t+1})$ 同样具有随机性。TD target 通常只包含一步真实随机转移，并用当前价值估计替代更长的随机奖励序列，因此**通常具有较低方差**，但并非只由一步奖励贡献方差。

### 为什么 TD 有偏差？

$V(S_{t+1})$ 是我们当前的**估计**（可能不准确），用它来计算 target 会引入偏差。
当训练收敛时 $V \to V^\pi$，TD 偏差 → 0。


## 3. 随机游走实验：MC vs TD(0)

### 环境：5 状态随机游走

```
L ← [0] — [1] — [2] — [3] — [4] → R
 ↑                                   ↑
左终止 (reward=0)                  右终止 (reward=1)
```

- 5 个非终止状态编号为 0, 1, 2, 3, 4
- 从中间状态 2 出发
- 每步 50% 概率向左/右移动
- 移出左边界 (state < 0) → reward=0, episode 终止
- 移出右边界 (state >= 5) → reward=1, episode 终止
- 真实价值: V(s) = [1/6, 2/6, 3/6, 4/6, 5/6]（对称随机游走的解析解）


In [ ]:
import numpy as np; import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed; set_seed(42)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

class RandomWalk:
    '''5 状态随机游走环境 (Gymnasium 5 元组接口)

    状态 0-4 (非终止), 左终止 = -1 (reward 0), 右终止 = 5 (reward 1)
    从中间状态 2 出发, 每步 50% 概率向左/右
    '''
    def __init__(self, n_states=5):
        self.n_states = n_states
        self.start = n_states // 2  # 从中间出发
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state, {}  # Gymnasium: (obs, info)

    def step(self, action=None):  # action 被忽略（随机环境）
        if np.random.rand() < 0.5:
            self.state -= 1  # 左
        else:
            self.state += 1  # 右

        if self.state < 0:  # 到达左终止 → reward=0
            return self.state, 0.0, True, False, {}
        elif self.state >= self.n_states:  # 到达右终止 → reward=1
            return self.state, 1.0, True, False, {}
        return self.state, 0.0, False, False, {}

rw = RandomWalk(n_states=5)
# 真实 V(s) = 从左到右: [1/6, 2/6, 3/6, 4/6, 5/6]
true_V = np.array([1/6, 2/6, 3/6, 4/6, 5/6])
print("真实 V(s):", true_V)
print("状态范围: 0 ~ 4 (非终止), 终止时 bootstrap=0")


In [ ]:
def td_zero_prediction(env, gamma, alpha, n_episodes):
    '''TD(0) 价值预测

    更新公式: V(S) ← V(S) + α[R + γ V(S') - V(S)]
    终止状态 bootstrap = 0（不引导）
    '''
    V = np.zeros(env.n_states)  # 只需要非终止状态 0~4
    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            next_state, reward, terminated, truncated, _ = env.step()
            done = terminated or truncated
            # TD(0) 更新：终止时 next_value=0
            next_value = 0.0 if terminated else V[next_state]
            td_error = reward + gamma * next_value - V[state]
            V[state] += alpha * td_error
            state = next_state
    return V  # 直接返回 V[0:n_states]

def mc_prediction(env, gamma, alpha, n_episodes):
    '''MC 价值预测 (Every-Visit)'''
    V = np.zeros(env.n_states)
    for ep in range(n_episodes):
        episode = []
        state, _ = env.reset(); done = False
        while not done:
            next_state, reward, terminated, truncated, _ = env.step()
            done = terminated or truncated
            episode.append((state, reward))
            state = next_state
        G = 0.0
        for state, reward in reversed(episode):
            G = reward + gamma * G
            V[state] += alpha * (G - V[state])
    return V

# 训练参数
n_runs, n_eps = 100, 100
alphas = [0.05, 0.1, 0.15, 0.2]
td_errors = {a: [] for a in alphas}
mc_errors = {a: [] for a in alphas}

for alpha in alphas:
    for run in range(n_runs):
        set_seed(run)
        rw_td = RandomWalk(5)
        rw_mc = RandomWalk(5)
        V_td = td_zero_prediction(rw_td, gamma=1.0, alpha=alpha, n_episodes=n_eps)
        V_mc = mc_prediction(rw_mc, gamma=1.0, alpha=alpha, n_episodes=n_eps)
        td_errors[alpha].append(np.sqrt(np.mean((V_td - true_V)**2)))
        mc_errors[alpha].append(np.sqrt(np.mean((V_mc - true_V)**2)))

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(alphas))
width = 0.35
td_means = [np.mean(td_errors[a]) for a in alphas]
mc_means = [np.mean(mc_errors[a]) for a in alphas]
td_stds = [np.std(td_errors[a]) for a in alphas]
mc_stds = [np.std(mc_errors[a]) for a in alphas]
ax.bar(x - width/2, td_means, width, yerr=td_stds, label='TD(0)', color='steelblue', capsize=3)
ax.bar(x + width/2, mc_means, width, yerr=mc_stds, label='MC', color='coral', capsize=3)
ax.set_xlabel('α (学习率)'); ax.set_ylabel('RMS Error')
ax.set_title(f'TD(0) vs MC ({n_runs} runs × {n_eps} episodes)')
ax.set_xticks(x); ax.set_xticklabels([str(a) for a in alphas])
ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig('outputs/figures/07_td_vs_mc.png'); plt.close()
print("✅ TD vs MC 对比图已保存")


**观察**：TD(0) 通常比 MC 收敛更快（因为利用了 bootstrapping 的结构信息）。

这就是 TD 的核心优势：**在样本有限时，利用 MDP 结构（Bellman 方程）比完全依赖采样更高效**。


## 4. n-Step TD：连接 MC 和 TD(0)

### 动机

- TD(0) 只用到 1 步后的信息（方差最低，偏差最高）
- MC 用到整个 episode（偏差最低，方差最高）
- **n-Step TD** 是它们之间的平滑插值

### n-Step Return

$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

- $n=1$：TD(0) target = $R_{t+1} + \gamma V(S_{t+1})$
- $n=\infty$（实际是整个 episode）：MC return = $G_t$

### n-Step TD 更新

$$V(S_t) \leftarrow V(S_t) + \alpha [G_t^{(n)} - V(S_t)]$$


In [ ]:
def n_step_td(env, gamma, alpha, n_step, n_episodes):
    '''n-step TD 预测'''
    V = np.zeros(env.n_states)  # 状态 0~4
    for ep in range(n_episodes):
        state, _ = env.reset()
        states, rewards = [state], [0.0]
        terminated = False
        T = float('inf')
        t = 0

        while True:
            if t < T:
                next_state, reward, term, trunc, _ = env.step()
                done = term or trunc
                states.append(next_state)
                rewards.append(reward)
                if done:
                    T = t + 1  # 记录终止时刻
                    terminated = term

            tau = t - n_step + 1  # 可以更新的最老状态
            if tau >= 0:
                # 计算 n-step return
                end = min(tau + n_step, T)
                G = sum(gamma**(i-tau-1) * rewards[i]
                        for i in range(tau+1, min(int(end)+1, len(rewards))))
                if tau + n_step < T:
                    # bootstrap from V
                    G += gamma**n_step * V[states[tau + n_step]]
                # 更新（只更新非终止状态）
                s = states[tau]
                if 0 <= s < env.n_states:
                    V[s] += alpha * (G - V[s])

            t += 1
            if tau == T - 1:
                break
            state = next_state
    return V

# 比较不同 n 值
n_values = [1, 2, 4, 8, 16, 32, 64, 128]
errors_by_n = {}
for n in n_values:
    run_errors = []
    for run in range(30):
        set_seed(run)
        rw = RandomWalk(5)
        V = n_step_td(rw, gamma=1.0, alpha=0.1, n_step=n, n_episodes=100)
        run_errors.append(np.sqrt(np.mean((V - true_V)**2)))
    errors_by_n[n] = np.mean(run_errors)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(n_values, [errors_by_n[n] for n in n_values], 'o-', linewidth=2)
ax.set_xlabel('n (步数)'); ax.set_ylabel('RMS Error')
ax.set_title('n-Step TD: 不同 n 值的表现')
ax.axhline(errors_by_n[128], color='r', linestyle='--', alpha=0.5, label=f'MC equivalent (n=128)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig('outputs/figures/07_nstep_td.png'); plt.close()
print("✅ n-step TD 对比图已保存")


## 5. TD(λ) 和 Eligibility Traces

### 核心思想

与其选一个固定的 $n$，不如**对所有 $n$ 加权平均**。

### λ-Return

$$G_t^\lambda = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

- $\lambda = 0$：TD(0)
- $\lambda = 1$：MC

权重是指数衰减的：较近的 n-step return 权重大，较远的权重小。

### Eligibility Trace

Eligibility trace 是一种**高效实现 TD(λ)** 的机制，不需要存储多条 n-step return。

直觉：每个状态有一个 "信用记录" $e(s)$，记录了它在最近的 episode 中被访问的频率（随时间衰减）。

**TD(λ) 前向视角更新**：$V(S_t) \leftarrow V(S_t) + \alpha [G_t^\lambda - V(S_t)]$

**TD(λ) 后向视角更新（用 eligibility trace）**：
- $e(s) \leftarrow \gamma\lambda e(s) + 1$（访问时增加）
- $V(s) \leftarrow V(s) + \alpha \delta_t \cdot e(s)$（所有状态的 V 都更新！）

其中 $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ 是 TD error。


## 6. 本节总结

### MC vs TD 对比

| 方面 | MC | TD(0) |
|------|-----|-------|
| Target | $G_t$（实际回报） | $R+\gamma V(S')$（bootstrap） |
| 更新时机 | Episode 结束 | 每步 |
| Bias | 0 | 有（来自不准确的 V） |
| Variance | 大 | 小 |
| 收敛速度 | 慢 | 快 |
| 需要完整 episode | 是 | 否（可用于 continuing tasks） |

### n-Step 家族

```
TD(0) ← n-Step TD → MC
  ↑                    ↑
低偏差/高方差     高偏差/低方差
```

### 关键理解

1. **TD = DP + MC**：像 DP 一样 bootstrap，像 MC 一样从经验学习
2. **TD error 是后续所有算法的基础**：SARSA, Q-Learning, Actor-Critic 都基于它
3. **n-Step 和 λ 提供了平衡 bias-variance 的连续谱**


## 7. 练习

1. 手算推导：证明当 $V=V^\pi$ 时，$\mathbb{E}[\delta_t | S_t=s] = 0$（TD error 的期望为零）
2. 在 RandomWalk 上比较 First-Visit MC、Every-Visit MC 和 TD(0) 的收敛速度
3. 实现带 eligibility trace 的 TD(λ)（后向视角）
4. 为什么 $\gamma=1.0$ 在 RandomWalk 中可以工作？（提示：episodic 任务）


---
*下一节：[08_sarsa_q_learning.ipynb](08_sarsa_q_learning.ipynb) — SARSA 与 Q-Learning*
